# CLIP for Chest X-Ray Classification
### Journal Club 2026: Section 2, A Shared Space Across Modalities

This interactive notebook demonstrates a CLIP-style image-text model for chest X-ray classification.

We align X-ray images with clinical text descriptions across three classes:
- **Normal**: no acute cardiopulmonary abnormality
- **Viral Pneumonia**: pneumonia-like airspace/perihilar opacities
- **COVID-19**: peripheral/bilateral opacities associated with COVID-19 pneumonia

By the end of this notebook you will:
1. Prepare the Kaggle COVID-19 Radiography Database
2. Pair each class with clinical text prompts
3. Build a compact CLIP-style model with a ResNet-18 image encoder and DistilBERT text encoder
4. Train image embeddings against class prompt embeddings
5. Evaluate zero-shot-style classification and visualize the shared embedding space

This notebook is intended for education and method exploration, not clinical decision-making.


---
## Section 1. Install Dependencies

In [ ]:
# Run this cell only if the packages are missing in your environment.
# It is safe to leave RUN_INSTALLS = False when running in an already prepared kernel.
RUN_INSTALLS = True

if RUN_INSTALLS:
    import subprocess
    import sys

    packages = [
        "kaggle",
        "transformers",
        "timm",
        "scikit-learn",
        "matplotlib",
        "seaborn",
        "umap-learn",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
else:
    print("Skipping installation. Set RUN_INSTALLS = True if imports fail.")


---
## Section 2. Kaggle Authentication

If the dataset is already downloaded, skip this section.

For Kaggle downloads, we use the new method using API keys, not the old token in a json file


In [ ]:
from google.colab import files

uploaded = files.upload()
print('Uploaded files:', list(uploaded.keys()))


import os
import shutil

os.makedirs('/root/.kaggle', exist_ok=True)

if os.path.exists('/content/kaggle.json'):
    shutil.copy('/content/kaggle.json', '/root/.kaggle/kaggle.json')
elif os.path.exists('/content/kaggle_API.txt'):
    shutil.copy('/content/kaggle_API.txt', '/root/.kaggle/access_token')
else:
    raise FileNotFoundError(
        'Upload kaggle.json or kaggle_API.txt first.'
    )

# os.chmod('/root/.kaggle/kaggle.json', 0o600)
# print('Kaggle token placed at /root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/access_token', 0o600)

---
## Section 3. Download the Chest X-Ray Dataset from Kaggle

We use the **COVID-19 Radiography Database** by Tawsifur Rahman et al., which contains:
- `Normal/` — healthy chest X-rays
- `Viral Pneumonia/` — classic pneumonia
- `COVID/` — COVID-19 positive cases

Kaggle dataset identifier: `tawsifurrahman/covid19-radiography-database`

Dataset page: https://www.kaggle.com/datasets/tawsifurrahman/covid19-radiography-database

> **Expected download size:** ~1.5 GB. This may take 2–5 minutes on Colab.

In [ ]:
! kaggle datasets list -s covid19-radiography-database | head

In [ ]:
!kaggle datasets download -d tawsifurrahman/covid19-radiography-database

---
## Section 4. Explore the Dataset

Let us inspect the folder structure and count images per class.

In [ ]:
# unzip the dataset

from pathlib import Path
import zipfile

ZIP_PATH = Path("/content/covid19-radiography-database.zip")
EXTRACT_DIR = Path("/content")

if not ZIP_PATH.exists():
    raise FileNotFoundError(f"Zip file not found: {ZIP_PATH}")

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zf.extractall(EXTRACT_DIR)

print(f"Extracted {ZIP_PATH.name} to {EXTRACT_DIR}")

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg


DATA_DIR = "/content/COVID-19_Radiography_Dataset"
DATA_DIR = Path(DATA_DIR)

CLASS_MAP = {
    "Normal": "Normal",
    "Viral Pneumonia": "Viral Pneumonia",
    "COVID": "COVID-19",
}

if not DATA_DIR.exists():
    raise FileNotFoundError(f"DATA_DIR does not exist: {DATA_DIR}")

print(f"Dataset root: {DATA_DIR.resolve()}")
print("Folder structure:")
for folder in sorted(DATA_DIR.iterdir()):
    img_folder = folder / "images"
    if img_folder.is_dir():
        n = len([p for p in img_folder.iterdir() if p.suffix.lower() in {".png", ".jpg", ".jpeg"}])
        print(f"  {folder.name}/images: {n:,} images")


In [ ]:
fig, axes = plt.subplots(1, len(CLASS_MAP), figsize=(12, 4))
fig.suptitle("One sample per class", fontsize=14)

for ax, (folder, label) in zip(axes, CLASS_MAP.items()):
    img_dir = DATA_DIR / folder / "images"
    images = sorted([p for p in img_dir.iterdir() if p.suffix.lower() in {".png", ".jpg", ".jpeg"}])
    if not images:
        ax.set_title(label + "\n(no images)")
        ax.axis("off")
        continue
    img = mpimg.imread(images[0])
    ax.imshow(img, cmap="gray")
    ax.set_title(label)
    ax.axis("off")

plt.tight_layout()
plt.show()


---
## Section 5. Define Clinical Text Prompts

CLIP learns to align images with free-form text descriptions. For each class we define a short clinical report-style prompt.

These prompts are used in two ways:
- **During training** — each image is paired with the prompt for its class as the positive text; the other two prompts are negatives.
- **During zero-shot inference** — we embed all three prompts and find the one closest to a test image.

You can experiment with different prompt wordings to see how text affects alignment.

In [ ]:
# Clinical text prompts for each class
# Feel free to modify these to experiment with prompt engineering.
CLASS_PROMPTS = {
    'Normal': (
        'Chest X-ray shows clear lung fields with no consolidation, '
        'no pleural effusion, and a normal cardiothoracic ratio. '
        'No acute cardiopulmonary abnormality identified.'
    ),
    'Viral Pneumonia': (
        'Chest X-ray demonstrates patchy airspace opacities consistent '
        'with viral pneumonia. Peribronchial thickening and perihilar '
        'infiltrates are present bilaterally.'
    ),
    'COVID': (
        'Chest X-ray reveals bilateral peripheral ground-glass opacities '
        'with a lower lobe predominance, findings typical of COVID-19 '
        'pneumonia.'
    ),
}

for label, prompt in CLASS_PROMPTS.items():
    print(f'[{label}]\n  {prompt}\n')

---
## Section 6. Build the Dataset and DataLoader

We build a `ChestXRayDataset` that:
- Loads images from all three class folders
- Applies standard image augmentation for training
- Returns `(image_tensor, prompt_text, label_index)` per sample

To keep training fast on Colab, we subsample up to **500 images per class** (1 500 total). Increase `MAX_PER_CLASS` for a fuller experiment.

In [ ]:
import random
from pathlib import Path

from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from sklearn.model_selection import train_test_split

MAX_PER_CLASS = 500
IMG_SIZE = 224
SEED = 42
BATCH_SIZE = 32
random.seed(SEED)
torch.manual_seed(SEED)

FOLDER_MAP = {
    "Normal": ("Normal", 0),
    "Viral Pneumonia": ("Viral Pneumonia", 1),
    "COVID": ("COVID", 2),
}

train_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def collect_samples(data_dir, folder_map, max_per_class=None):
    samples = []
    for folder, (label_str, label_idx) in folder_map.items():
        img_dir = Path(data_dir) / folder / "images"
        if not img_dir.is_dir():
            raise FileNotFoundError(f"Missing expected folder: {img_dir}")
        paths = sorted([p for p in img_dir.iterdir() if p.suffix.lower() in {".png", ".jpg", ".jpeg"}])
        if max_per_class is not None:
            rng = random.Random(SEED + label_idx)
            paths = rng.sample(paths, min(max_per_class, len(paths)))
        samples.extend([(str(path), label_str, label_idx) for path in paths])
    return samples


class ChestXRayDataset(Dataset):
    """Returns image tensor, class prompt text, and label index."""

    def __init__(self, samples, prompts, transform=None):
        self.samples = list(samples)
        self.prompts = prompts
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label_str, label_idx = self.samples[idx]
        image = Image.open(path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, self.prompts[label_str], label_idx


samples = collect_samples(DATA_DIR, FOLDER_MAP, max_per_class=MAX_PER_CLASS)
labels = [label_idx for _, _, label_idx in samples]
train_samples, val_samples = train_test_split(
    samples,
    test_size=0.2,
    random_state=SEED,
    stratify=labels,
)

train_ds = ChestXRayDataset(train_samples, CLASS_PROMPTS, transform=train_transform)
val_ds = ChestXRayDataset(val_samples, CLASS_PROMPTS, transform=val_transform)

num_workers = 2 if torch.cuda.is_available() else 0
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=num_workers)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=num_workers)

print(f"Total: {len(samples):,} | Train: {len(train_ds):,} | Val: {len(val_ds):,}")


---
## Section 7. Build the CLIP Model

Our simplified CLIP has two towers:

| Tower | Backbone | Output |
|---|---|---|
| **Image encoder** | ResNet-18 (pretrained on ImageNet) | 512-d → projection head → 256-d embedding |
| **Text encoder** | DistilBERT (pretrained on general text) | CLS token → projection head → 256-d embedding |

Both embeddings are L2-normalised before computing the contrastive loss. The projection heads (two-layer MLP) allow the shared space dimensionality to be freely chosen independently of the backbone.

A learnable **temperature** parameter `τ` scales the logits in the InfoNCE loss, as in the original CLIP paper.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from transformers import DistilBertModel, DistilBertTokenizer

EMBED_DIM = 256
if torch.cuda.is_available():
    device = "cuda"
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"Using device: {device}")


class ProjectionHead(nn.Module):
    def __init__(self, in_dim, out_dim=EMBED_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, in_dim),
            nn.GELU(),
            nn.Linear(in_dim, out_dim),
        )

    def forward(self, x):
        return self.net(x)


class ImageEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        try:
            weights = models.ResNet18_Weights.DEFAULT
            backbone = models.resnet18(weights=weights)
        except AttributeError:
            backbone = models.resnet18(pretrained=True)
        in_features = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.backbone = backbone
        self.projection = ProjectionHead(in_features)

    def forward(self, x):
        features = self.backbone(x)
        embeddings = self.projection(features)
        return F.normalize(embeddings, dim=-1)


class TextEncoder(nn.Module):
    def __init__(self, model_name="distilbert-base-uncased"):
        super().__init__()
        self.tokenizer = DistilBertTokenizer.from_pretrained(model_name)
        self.bert = DistilBertModel.from_pretrained(model_name)
        self.projection = ProjectionHead(self.bert.config.hidden_size)

    def forward(self, texts):
        target_device = next(self.parameters()).device
        encoded = self.tokenizer(
            list(texts),
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt",
        ).to(target_device)
        output = self.bert(**encoded)
        cls = output.last_hidden_state[:, 0, :]
        embeddings = self.projection(cls)
        return F.normalize(embeddings, dim=-1)


class CLIPModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.image_encoder = ImageEncoder()
        self.text_encoder = TextEncoder()
        self.logit_scale = nn.Parameter(torch.ones([]) * 2.6593)

    def encode_image(self, images):
        return self.image_encoder(images)

    def encode_text(self, texts):
        return self.text_encoder(texts)

    def forward(self, images, texts):
        image_embeddings = self.encode_image(images)
        text_embeddings = self.encode_text(texts)
        scale = self.logit_scale.exp().clamp(max=100)
        return scale * image_embeddings @ text_embeddings.T


model = CLIPModel().to(device)
print("Model ready.")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


---
## Section 8. Define Contrastive and Class-Prompt Losses

The classic CLIP InfoNCE loss assumes every image-text pair in a batch is unique. In this teaching dataset, many images share the same class prompt, so ordinary batch InfoNCE would incorrectly treat same-class examples as negatives.

We keep `infonce_loss` for reference, but train with `class_prompt_loss`: each image is compared against the three class prompts, and the correct class prompt is the target. This keeps the CLIP idea while matching the class-labeled dataset.


In [ ]:
import torch
import torch.nn.functional as F


def infonce_loss(logits):
    """Symmetric InfoNCE for batches with unique image-text pairs."""
    n = logits.size(0)
    labels = torch.arange(n, device=logits.device)
    loss_i2t = F.cross_entropy(logits, labels)
    loss_t2i = F.cross_entropy(logits.T, labels)
    return (loss_i2t + loss_t2i) / 2


def class_prompt_loss(model, images, labels, class_prompts):
    """Cross-entropy over class text prompts."""
    image_embeddings = model.encode_image(images)
    text_embeddings = model.encode_text(class_prompts)
    scale = model.logit_scale.exp().clamp(max=100)
    logits = scale * image_embeddings @ text_embeddings.T
    return F.cross_entropy(logits, labels), logits

print("Loss functions defined.")


---
## Section 9. Train the CLIP-Style Model

We use AdamW with separate learning rates for the image encoder and text encoder. Training compares every image to the three class prompts and optimizes the correct class prompt.

For a short journal-club demo, set `MAX_EPOCHS = 2` or `3`. For a better result, increase the number of epochs and `MAX_PER_CLASS`.


In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

MAX_EPOCHS = 5
CLASS_LABELS = list(CLASS_PROMPTS.keys())
CLASS_TEXTS = [CLASS_PROMPTS[label] for label in CLASS_LABELS]

text_params = list(model.text_encoder.parameters())
other_params = list(model.image_encoder.parameters()) + [model.logit_scale]

optimizer = AdamW([
    {"params": other_params, "lr": 1e-4, "weight_decay": 1e-4},
    {"params": text_params, "lr": 1e-5, "weight_decay": 1e-2},
])
scheduler = CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)

train_losses, val_losses, val_accs = [], [], []

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    train_loss_sum = 0.0
    for images, _, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        loss, _ = class_prompt_loss(model, images, labels, CLASS_TEXTS)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss_sum += loss.item() * images.size(0)

    train_loss = train_loss_sum / len(train_loader.dataset)

    model.eval()
    val_loss_sum = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, _, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            loss, logits = class_prompt_loss(model, images, labels, CLASS_TEXTS)
            val_loss_sum += loss.item() * images.size(0)
            correct += (logits.argmax(dim=-1) == labels).sum().item()
            total += labels.numel()

    val_loss = val_loss_sum / len(val_loader.dataset)
    val_acc = correct / max(total, 1)
    scheduler.step()

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
        f"val_acc={val_acc:.3f} | scale={model.logit_scale.exp().item():.2f}"
    )

print("Training complete.")


---
## Section 10. Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(train_losses, label="Train loss", marker="o")
ax1.plot(val_losses, label="Val loss", marker="s")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend(loc="upper left")

if val_accs:
    ax2 = ax1.twinx()
    ax2.plot(val_accs, label="Val accuracy", color="seagreen", marker="^")
    ax2.set_ylabel("Accuracy")
    ax2.set_ylim(0, 1)
    ax2.legend(loc="upper right")

plt.title("CLIP-Style Training Curve: Chest X-Ray")
plt.tight_layout()
plt.show()


---
## Section 11. Zero-Shot Classification

This is the key capability introduced by CLIP: **no task-specific fine-tuning is needed**.

For each test image we:
1. Encode the image → image embedding
2. Encode all three class prompts → three text embeddings
3. Compute cosine similarity between the image and each text
4. The predicted class is the one with the highest similarity

We then compute a classification report and confusion matrix.

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

CLASS_NAMES = ["Normal", "Viral Pneumonia", "COVID-19"]

model.eval()
with torch.no_grad():
    text_embeddings = model.encode_text(CLASS_TEXTS)

all_preds, all_true = [], []
with torch.no_grad():
    for images, _, labels in val_loader:
        images = images.to(device)
        image_embeddings = model.encode_image(images)
        similarities = image_embeddings @ text_embeddings.T
        preds = similarities.argmax(dim=-1).cpu().numpy()
        all_preds.extend(preds)
        all_true.extend(labels.numpy())

print("=== Prompt-Based Classification Report ===")
print(classification_report(all_true, all_preds, target_names=CLASS_NAMES, zero_division=0))

cm = confusion_matrix(all_true, all_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Prompt-Based Confusion Matrix")
plt.tight_layout()
plt.show()


---
## Section 12. Visualise the Shared Embedding Space (UMAP)

We project all validation image embeddings and the three text prompt embeddings into 2D using **UMAP**.

What to look for:
- Image embeddings from the same class should cluster together.
- Each text prompt embedding (★) should sit near its corresponding image cluster — this is the hallmark of a well-trained shared embedding space.

In [ ]:
# UMAP is optional. Run this only if umap-learn is not already installed.
RUN_UMAP_INSTALL = False

if RUN_UMAP_INSTALL:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "umap-learn"])
else:
    print("Skipping UMAP installation. Set RUN_UMAP_INSTALL = True if import umap fails.")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

try:
    import umap
except ImportError as exc:
    raise ImportError("Install umap-learn or run the previous cell with RUN_UMAP_INSTALL = True.") from exc

model.eval()
img_embs_all, labels_all = [], []

with torch.no_grad():
    for images, _, labels in val_loader:
        emb = model.encode_image(images.to(device))
        img_embs_all.append(emb.cpu().numpy())
        labels_all.extend(labels.numpy())

img_embs_all = np.vstack(img_embs_all)
labels_all = np.array(labels_all)

with torch.no_grad():
    txt_embs_np = model.encode_text(CLASS_TEXTS).cpu().numpy()

combined = np.vstack([img_embs_all, txt_embs_np])
reducer = umap.UMAP(n_components=2, random_state=SEED)
coords = reducer.fit_transform(combined)

img_coords = coords[:len(img_embs_all)]
txt_coords = coords[len(img_embs_all):]

colors = ["steelblue", "darkorange", "seagreen"]
fig, ax = plt.subplots(figsize=(8, 6))

for idx, (name, color) in enumerate(zip(CLASS_NAMES, colors)):
    mask = labels_all == idx
    ax.scatter(img_coords[mask, 0], img_coords[mask, 1], c=color, alpha=0.4, s=15, label=name)

for idx, (name, color) in enumerate(zip(CLASS_NAMES, colors)):
    ax.scatter(
        txt_coords[idx, 0],
        txt_coords[idx, 1],
        c=color,
        marker="*",
        s=400,
        edgecolors="black",
        linewidths=0.8,
        zorder=5,
        label=f"{name} prompt",
    )

ax.set_title("UMAP of Shared CLIP Embedding Space")
ax.set_xlabel("UMAP-1")
ax.set_ylabel("UMAP-2")
ax.legend(loc="best", fontsize=8)
plt.tight_layout()
plt.show()


---
## Section 13. Save the Trained Model

Save the model weights to Google Drive or your local machine for future use.

In [ ]:
from pathlib import Path
import torch

SAVE_PATH = Path("clip_chest_xray.pt")
torch.save({
    "model_state_dict": model.state_dict(),
    "class_prompts": CLASS_PROMPTS,
    "class_labels": CLASS_LABELS,
    "embed_dim": EMBED_DIM,
}, SAVE_PATH)
print(f"Model checkpoint saved to {SAVE_PATH.resolve()}")


---
## Section 14. Discussion Points for Journal Club

Use these questions to guide the group discussion:

1. **Prompt sensitivity** — How does changing the wording of the clinical prompts affect zero-shot accuracy? Try replacing the detailed clinical descriptions with short labels like `"Normal chest X-ray"` and compare.

2. **Shared embedding space** — In the UMAP plot, do the text prompt embeddings (★) fall inside the corresponding image clusters? What does it mean if they do not?

3. **Domain gap** — We used general pretrained weights (ImageNet ResNet-18, general DistilBERT). How would performance change if we used a domain-specific backbone like BiomedCLIP, which was pretrained on 15 million biomedical figure–caption pairs?

4. **Data efficiency** — CLIP-style training typically needs millions of pairs. Here we used only ~1 500 images. What are the implications for clinical deployment?

5. **Symmetric loss** — The InfoNCE loss is symmetric (image→text and text→image). Why is symmetry important? What would happen with a one-sided loss?

6. **Zero-shot vs. fine-tuning** — When would you choose zero-shot CLIP over a fully supervised classifier for medical imaging tasks?